In [ ]:
%matplotlib qt

import numpy as np
import diffpy.structure
import pyxem as pxm
import hyperspy.api as hs

accelarating_voltage = 100  # kV
camera_length = 0.2  # m
import matplotlib.pyplot as plt
import copy
from hyperspy.signals import Signal2D
import os
import glob
import gc

from scipy.ndimage import generic_filter
Diffraction_Calibration = 0.0063345*2*1.03 #/2.35

In [ ]:
Raw_Data = hs.load(r"D:\STEM\Sample C\STEM 16-502b3-CameraImageSeries.hspy", reader="hspy")
#Raw_Data = Raw_Data.rebin(scale=(1, 1, 2, 2))
#Normalise the data
Raw_Data.data = Raw_Data.data.astype('float32')

#Bool = Raw_Data.data > Raw_Data.data.mean() + 5*Raw_Data.data.std()
#Raw_Data.data[Bool] = Raw_Data.data.mean()

Raw_Data.data *= 1 / Raw_Data.data.max()

#Select the region, widget working!
reg = hs.roi.RectangularROI(left=-100, top=-10, right=600, bottom=300)
Raw_Data.plot(cmap='inferno', vmax=0.1)
reg.add_widget(Raw_Data)

In [ ]:
total_intensity = Raw_Data.data.sum(axis=(-2, -1))

median = np.median(total_intensity)
mad = np.median(np.abs(total_intensity - median))

threshold = median + 10 * mad
bad_mask = total_intensity > threshold

mean_dp = Raw_Data.data[~bad_mask].mean(axis=0)

# Replace bad pixels
Raw_Data.data[bad_mask] = mean_dp

In [ ]:
Raw_Data.data *= 1 / Raw_Data.data.max()
#Select the region, widget working!
reg = hs.roi.RectangularROI(left=-100, top=-10, right=300, bottom=300)
Raw_Data.plot(cmap='inferno', vmax=0.1)
reg.add_widget(Raw_Data)

In [ ]:
#Array = np.array(Raw_Data)

#Scan = Array.sum(axis=(-1, -2)) 

#Boolean = Raw_Data > 0.002
#Array[Boolean] = 0

#Raw_Data = Signal2D(Array)

#Raw_Data.plot()

In [ ]:
##################                    #####################
##################    Callibration    #####################
##################                    #####################
Cropped_Data = reg(Raw_Data)

Diffraction_Calibration = 0.0063345*2*1.03 #/2.35

#Remove the old data to save memory
del Raw_Data

Cropped_Data.set_signal_type('electron_diffraction')  # assign electron diffraction so pyxem can work
Cropped_Data.set_diffraction_calibration(Diffraction_Calibration)
gc.collect() #helps manage RAM


##################                      #####################
##################    Pre-Processing    #####################
##################                      #####################

from pyxem.utils.diffraction import investigate_dog_background_removal_interactive

from skimage import filters

def crop_minimum(image, minimum=0.0005):
    copied = image.copy()
    copied[copied <= minimum] = 0.
    return copied



Filtered_Data = Cropped_Data.inav[10:20,10:20].deepcopy()
Filtered_Data = Filtered_Data.subtract_diffraction_background(
    "difference of gaussians",
    min_sigma=1,   #Test values here
    max_sigma=20,
)

#s_filtered_h = s.subtract_diffraction_background("h-dome", inplace=False, h=0.7)


hs.plot.plot_images(
    [Cropped_Data.inav[0, 0], Filtered_Data.inav[0, 0]],
    label=["Original", "Difference of Gaussians"],
    tight_layout=True,
    norm="symlog",
    cmap="inferno_r",
    colorbar=None,
    vmin=0.,vmax=0.01
)

In [ ]:
# remove the background intensity with difference of gaussian
Filtered_Data = Cropped_Data.subtract_diffraction_background(method="difference of gaussians",
                                                            min_sigma=1.,
                                                            max_sigma=20., ) #need to check the min_sigma and max_sigma with #1 and 40 baseline 
                                                                            #background removal script before running this
# smooth out the output
Filtered_Data = Filtered_Data.map(filters.gaussian, sigma=0.5, inplace=False)
# remove low intensities
Filtered_Data = Filtered_Data.map(crop_minimum, minimum = 0.0001, inplace=False)
Filtered_Data.set_signal_type(signal_type="electron_diffraction")
Filtered_Data.plot(cmap = 'inferno_r', vmax=0.1)

#Plot a zoomed in image of the direct beam diffraction spot
#Filtered_Data.isig[256/2-15:256/2+15,256/2-15:256/2+15].plot(cmap = 'inferno')

#Center the direct beam
Filtered_Data.center_direct_beam(method='center_of_mass',half_square_width=15)
Filtered_Data.set_scan_calibration(8.964214570007414E-9)

gc.collect() #helps manage RAM

##################                       #####################
##################    Mask Generation    #####################
##################                       #####################

from hyperspy.signals import Signal2D
from pyxem.utils.diffraction import circular_mask

Shape = Filtered_Data.axes_manager.signal_shape
Mask = np.ones(Shape)
Data_Array = np.array(Filtered_Data)

#Uniformity Mask
Data_Array = np.array(Filtered_Data)

Data_Array[Data_Array < 0.002] = 0 #0.002
Data_Array[Data_Array >= 0.002] = 1

Masked_Data = Data_Array*Mask
Masked_Data = Signal2D(Masked_Data)
Masked_Data.plot(vmax=0.01)

In [ ]:
#Filtered_Data = Masked_Data
#del Masked_Data
#del Cropped_Data
Filtered_Data  = reg(Raw_Data)
#Diffraction_Calibration = 0.0063345*2*1.03

##################                         #####################
##################    Template Matching    #####################
##################                         #####################

from diffsims.generators.diffraction_generator import DiffractionGenerator
from diffsims.generators.rotation_list_generators import get_beam_directions_grid
from diffsims.libraries.structure_library import StructureLibrary

from diffsims.generators.library_generator import DiffractionLibraryGenerator

from diffsims.generators.zap_map_generator import get_rotation_from_z_to_direction
from diffsims.generators.rotation_list_generators import get_grid_around_beam_direction

from pyxem.generators.indexation_generator import AcceleratedIndexationGenerator
from pyxem.utils.indexation_utils import results_dict_to_crystal_map

structure_FeSn = diffpy.structure.loadStructure(r"C:\Users\pycbr\Documents\PhD\FeSn_1.cif")
structure_Fe3Sn2 = diffpy.structure.loadStructure(r"C:\Users\pycbr\Documents\PhD\Fe3Sn2_1.cif")

half_shape = (Filtered_Data.data.shape[-2]//2, Filtered_Data.data.shape[-1]//2)
reciprocal_radius = np.sqrt(half_shape[0]**2 + half_shape[1]**2)*Diffraction_Calibration#*0.2  #Set radius for which spots will be simualted out to

Diff_Gen = DiffractionGenerator(accelerating_voltage=100,
                                precession_angle=0.0,
                                scattering_params='xtables',
                                shape_factor_model="lorentzian",
                                minimum_intensity=0.0001,) #What minimimum intensity is included in pattern

Lib_Gen = DiffractionLibraryGenerator(Diff_Gen)

#These are the euler angles for the rotations of the crystal
#u, v, w = 2,1,0
#selected_rot = get_rotation_from_z_to_direction(structure_Fe3Sn2,(u, v, w))
#get_rotation_from_z_to_direction(structure_Fe3Sn2,(u, v, w))
#(0,selected_rot[1],selected_rot[2])

rot_list_hex = np.array([[0.0, 90.0, -90.0],])
rot_list_cubic = np.array([[0.0, 90.0, -90.0],])
grid_cub2 = get_beam_directions_grid("trigonal", resolution=2.5, mesh="spherified_cube_corner")

Struc_Lib = StructureLibrary(['FeSn','Fe3Sn2'],
                             [structure_FeSn,structure_Fe3Sn2],
                             [rot_list_hex, rot_list_cubic])

Diff_Lib = Lib_Gen.get_diffraction_library(Struc_Lib,
                                           calibration=Diffraction_Calibration,
                                           reciprocal_radius=reciprocal_radius,
                                           half_shape=half_shape,
                                           with_direct_beam=False,
                                           max_excitation_error=0.03) # See notes on max excitation errror

#Normalise the simulations
#Diff_Lib['FeSn']['intensities'][0] =  1000*Diff_Lib['FeSn']['intensities'][0]*(Diff_Lib['Fe3Sn2']['intensities'][0][35] / Diff_Lib['FeSn']['intensities'][0][26])


from pyxem.utils import indexation_utils as iutls
from pyxem.utils import plotting_utils as putls
from pyxem.utils import polar_transform_utils as ptutls
from pyxem.utils import expt_utils as eutls

#Image a particualar diffraction pattern simulation 

Test_Spots = Diff_Lib["FeSn"]['simulations'][0]
Image_Size = 256
Background = np.ones([Image_Size,Image_Size])

i = int(Image_Size/2) - 2
j = int(Image_Size/2) - 2
while i < Image_Size/2 + 3:
    while j < Image_Size/2 + 3:
        Background[i,j] = 0
        j = j + 1
    j = int(Image_Size/2) - 2
    i = i + 1

putls.plot_template_over_pattern(Background,
                                 Test_Spots,
                                 size_factor = 10,
                                direct_beam_position=(256/2,256/2),)

In [ ]:
# pull out a user selected image and simulation
image = Filtered_Data.inav[1,1].data
simulation_test = Diff_Lib["FeSn"]["simulations"][0]##[-8]

# for completeness in the illustration, all keyword arguments are given and explained
# an array of angles and corresponding correlation values are returned
a, c = iutls.get_in_plane_rotation_correlation(
    image,
    simulation_test,
    intensity_transform_function=None,  # a function applied both to the image and template intensities before calculating the correlation
    delta_r = 1.,                        # sampling in the radial direction
    delta_theta = 0.1,                  # sampling in the azimuthal direction
    max_r = 50,                       # maximum radius to consider, by default the distance from the center to the corner
    find_direct_beam = False,            # convenience, if the pattern was not centered, this will perform a rough centering
    direct_beam_position = (256 / 2, 256 / 2),    # manually provide the coordinates of the direct beam
    normalize_image=True,               # divide the correlation by the norm of the image
    normalize_template=True,            # divide the correlation by the norm of the template
)

###########################################################################

# Rolling Average Calculation, slightly improves accuracy of rotation angle of the pattern:

d = copy.deepcopy(c)

Rolling_Average_N = 80  #Rolling Average is done over 2*N + 1 points
i = Rolling_Average_N
while i < len(c)-Rolling_Average_N:
    d[i] = sum(c[i-Rolling_Average_N : i + Rolling_Average_N + 1])/(2*Rolling_Average_N + 1)
    i = i + 1

###########################################################################

fig, ax = plt.subplots()
ax.plot(a, c, label = 'Raw Correlation')
ax.plot(a, d, label = 'Smoothed Correlation' )
ax.set_xlim(0, 360)
ax.set_xlabel("Angular shift (Degrees)", fontsize = 20, labelpad = 20)
ax.set_ylabel("Correlation", fontsize = 20, labelpad = 20 )

ax.tick_params(labelsize = 20)

fig.legend(bbox_to_anchor = (0.90,0.88), prop={'size': 20})

#Combine image and simulation
putls.plot_template_over_pattern(image,                           #Diffraction pattern
                                 simulation_test,                 #Simulated Diffraction pattern
                                 in_plane_angle= a[np.argmax(d)],  #Apply an in plane rotation to the template, selected from max of the correlation
                                 coordinate_system = "cartesian",     #Which coordinate system to plot image and template in
                                 marker_color = "red",
                                 marker_type = '2',#"H",
                                 size_factor = 30,                #Scaling factor for simulated crosses
                                 vmax=0.005,                     #
                                 max_r = None,
                                 find_direct_beam=False,
                                 direct_beam_position=(256 /2 ,256 /2),
                                 cmap = "inferno_r")

In [ ]:
##################                               #####################
##################    Multi-Template Matching    #####################
##################                               ##################### 

simulations = Diff_Lib["FeSn"]["simulations"]
delta_r = 1.0
delta_theta = 1.0 /2
max_r = 156
intensity_transform_function = np.sqrt
find_direct_beam = False
direct_beam_position = (256/2,256/2)
normalize_image = True
normalize_templates = False

gc.collect()

##################                                     #####################
##################    Multi-Template Multi-Matching    #####################
##################                                     #####################

Copied_Filtered_Data = copy.deepcopy(Filtered_Data.data)
Copied_Filtered_Data  = hs.signals.Signal2D(Copied_Filtered_Data)
Copied_Filtered_Data .set_signal_type('electron_diffraction')
Copied_Filtered_Data .set_diffraction_calibration(Diffraction_Calibration)
Copied_Filtered_Data.as_lazy() #convert to lazy dataset


frac_keep = 1
n_keep = None  #The ultimate correlation step
indexes, angles, corrs, angles_m, corrs_m = iutls.correlate_library_to_pattern(
    image, simulations, frac_keep, n_keep, delta_r, delta_theta, max_r,
    intensity_transform_function, find_direct_beam, direct_beam_position,
    normalize_image, normalize_templates,)

#Generate the results matrix
            

result, phasedict = iutls.index_dataset_with_template_rotation(Filtered_Data,
                                                    Diff_Lib,
                                                      # if we have multiple phases we can also specify which ones we want to consider. If it's not specified, all phases are used.
                                                    n_best = 3,
                                                    frac_keep = 1,
                                                    n_keep = n_keep,
                                                    delta_r = delta_r,
                                                    delta_theta = delta_theta,
                                                    max_r = 115,
                                                    intensity_transform_function=intensity_transform_function,
                                                    normalize_images = normalize_image,
                                                    normalize_templates=normalize_templates,
                                                    
                                                    )

indexer = AcceleratedIndexationGenerator(Copied_Filtered_Data, Diff_Lib)
Number_Of_Tests = 100
indexation_results,phase_dict  = indexer.correlate(n_largest=Number_Of_Tests)


#Percentage of N_best results that are Fe3Sn2
Volume = (indexation_results['phase_index'].shape[0]*indexation_results['phase_index'].shape[1]*indexation_results['phase_index'].shape[2])

Percentage = (sum(sum(sum(indexation_results['phase_index']))) / Volume )*100

Copied_Filtered_Data.plot(cmap='inferno',vmax=0.001)
Copied_Filtered_Data.set_scan_calibration(3)

In [ ]:
Crystal_Map =  2*( indexation_results['phase_index'][:,:,0] -0.5)
RawIndexMap = Signal2D(indexation_results['phase_index'][:,:,0])

A = Signal2D(Crystal_Map)
Correlation_Map = indexation_results['correlation'][:,:,0]
A.plot(cmap = 'inferno')

CorrAverage = np.mean(Correlation_Map)
CorrStdDev = np.std(Correlation_Map)

Bool = Correlation_Map < CorrAverage - CorrStdDev

Crystal_Map_Copy = Crystal_Map
Crystal_Map[Bool] = 0

A = Signal2D(Crystal_Map)
A.plot(cmap = 'Set1')

# Calculate the confidence map for each phase from the correlation value for the first occurance of each phase within the phase index

Confidence_Map = np.ones([indexation_results['phase_index'].shape[0],indexation_results['phase_index'].shape[1]])  #Initialise Map with all 1s
Correlation_Values = indexation_results['correlation']
i = 0
j = 0

while i < indexation_results['phase_index'].shape[0]:
    while j < indexation_results['phase_index'].shape[1]:
        
        Phase_Column = indexation_results['phase_index'][i,j,:]
        Zero_Index = np.where(indexation_results['phase_index'][0,0,:] == 0)[0][0]  #Find the first 0 in the column
        One_Index = np.where(indexation_results['phase_index'][0,0,:] == 1)[0][0]   #Find the first 1 in the column

        if Zero_Index > One_Index:
            Confidence_Map[i,j] = 1 - ( Correlation_Values[i,j,Zero_Index] / Correlation_Values[i,j,One_Index] )
        else:
            Confidence_Map[i,j] = (1 - ( Correlation_Values[i,j,One_Index] / Correlation_Values[i,j,Zero_Index] ))
        j = j + 1
    i = i + 1
    j = 0

        
Confidence_Map = Confidence_Map * 100

ConfAverage = np.mean(Confidence_Map)
ConfStdDev = np.std(Confidence_Map)

Bool2 = Confidence_Map < ConfAverage - 2*ConfStdDev
Crystal_Map_Copy[Bool2] = 0
D = Signal2D(Crystal_Map_Copy)
D.plot(cmap = 'Set1')

B = Signal2D(Correlation_Map)
C = Signal2D(Confidence_Map)

Copied_Filtered_Data.save(r"D:\STEM\Repeat\Mixed STEM 4\Processed")
A.save(r"D:\STEM\Repeat\Mixed STEM 4\Final_Map")
RawIndexMap.save(r"D:\STEM\Repeat\Mixed STEM 4\Index_Map")
B.save(r"D:\STEM\Repeat\Mixed STEM 4\Correlation_Map")
C.save(r"D:\STEM\Repeat\Mixed STEM 4\Confidence_Map")